# Manhattan Taxi Hotspots Map
Interactive heatmaps of top 500 pickup and dropoff locations in Manhattan, sized by trip count.

In [0]:
pickup_df = spark.sql("""
SELECT dl.latitude, dl.longitude, COUNT(*) AS trip_count
FROM students_data.`team3-taxi`.fact_trip ft
INNER JOIN students_data.`team3-taxi`.dim_location dl ON ft.pickup_location_key = dl.location_key
WHERE dl.latitude BETWEEN 40.70 AND 40.88
  AND dl.longitude BETWEEN -74.02 AND -73.90
GROUP BY dl.latitude, dl.longitude
ORDER BY trip_count DESC
LIMIT 500
""").toPandas()
print(f"Pickup locations: {len(pickup_df)}")
pickup_df.head()

In [0]:
dropoff_df = spark.sql("""
SELECT dl.latitude, dl.longitude, COUNT(*) AS trip_count
FROM students_data.`team3-taxi`.fact_trip ft
INNER JOIN students_data.`team3-taxi`.dim_location dl ON ft.dropoff_location_key = dl.location_key
WHERE dl.latitude BETWEEN 40.70 AND 40.88
  AND dl.longitude BETWEEN -74.02 AND -73.90
GROUP BY dl.latitude, dl.longitude
ORDER BY trip_count DESC
LIMIT 500
""").toPandas()
print(f"Dropoff locations: {len(dropoff_df)}")
dropoff_df.head()

In [0]:
%pip install folium -q

In [0]:
import folium
from folium.plugins import HeatMap

m_pickup = folium.Map(location=[40.78, -73.97], zoom_start=13, tiles="CartoDB positron")

heat_data = [[row.latitude, row.longitude, row.trip_count] for _, row in pickup_df.iterrows()]
HeatMap(heat_data, radius=12, blur=8, max_zoom=15, name="Pickup Heatmap").add_to(m_pickup)

for _, row in pickup_df.head(20).iterrows():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=max(3, row.trip_count / pickup_df.trip_count.max() * 20),
        color="#e74c3c",
        fill=True,
        fill_opacity=0.7,
        tooltip=f"Trips: {row.trip_count:,}"
    ).add_to(m_pickup)

folium.LayerControl().add_to(m_pickup)
displayHTML(m_pickup._repr_html_())

In [0]:
import folium
from folium.plugins import HeatMap

m_dropoff = folium.Map(location=[40.78, -73.97], zoom_start=13, tiles="CartoDB positron")

heat_data = [[row.latitude, row.longitude, row.trip_count] for _, row in dropoff_df.iterrows()]
HeatMap(heat_data, radius=12, blur=8, max_zoom=15, name="Dropoff Heatmap").add_to(m_dropoff)

for _, row in dropoff_df.head(20).iterrows():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=max(3, row.trip_count / dropoff_df.trip_count.max() * 20),
        color="#3498db",
        fill=True,
        fill_opacity=0.7,
        tooltip=f"Trips: {row.trip_count:,}"
    ).add_to(m_dropoff)

folium.LayerControl().add_to(m_dropoff)
displayHTML(m_dropoff._repr_html_())

In [0]:
routes_df = spark.sql("""
SELECT
  ROUND(pl.latitude, 3) AS pickup_lat, ROUND(pl.longitude, 3) AS pickup_lng,
  ROUND(dl.latitude, 3) AS dropoff_lat, ROUND(dl.longitude, 3) AS dropoff_lng,
  COUNT(*) AS trip_count,
  ROUND(AVG(ft.trip_distance), 2) AS avg_distance_mi,
  ROUND(AVG(ft.trip_duration_minutes), 1) AS avg_duration_min
FROM students_data.`team3-taxi`.fact_trip ft
INNER JOIN students_data.`team3-taxi`.dim_location pl ON ft.pickup_location_key = pl.location_key
INNER JOIN students_data.`team3-taxi`.dim_location dl ON ft.dropoff_location_key = dl.location_key
WHERE ft.trip_distance > 0.3
  AND ft.trip_duration_minutes > 1
GROUP BY ROUND(pl.latitude, 3), ROUND(pl.longitude, 3), ROUND(dl.latitude, 3), ROUND(dl.longitude, 3)
HAVING NOT (ROUND(pl.latitude, 3) = ROUND(dl.latitude, 3) AND ROUND(pl.longitude, 3) = ROUND(dl.longitude, 3))
ORDER BY trip_count DESC
LIMIT 5
""").toPandas()
print(f"Top {len(routes_df)} journeys (whole database, ~100m zones):")
routes_df

In [0]:
import folium
import numpy as np

# Centre on the routes
all_lats = list(routes_df.pickup_lat) + list(routes_df.dropoff_lat)
all_lngs = list(routes_df.pickup_lng) + list(routes_df.dropoff_lng)
m_routes = folium.Map(
    location=[np.mean(all_lats), np.mean(all_lngs)],
    zoom_start=13, tiles="CartoDB positron"
)

colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
max_trips = routes_df.trip_count.max()

for i, row in routes_df.iterrows():
    color = colors[i % len(colors)]
    weight = 3 + (row.trip_count / max_trips) * 7  # thicker = more trips

    # Draw straight route line
    folium.PolyLine(
        locations=[[row.pickup_lat, row.pickup_lng], [row.dropoff_lat, row.dropoff_lng]],
        color=color, weight=weight, opacity=0.85,
        tooltip=(
            f"<b>Route #{i+1}</b><br>"
            f"Trips: {row.trip_count:,}<br>"
            f"Avg distance: {row.avg_distance_mi} mi<br>"
            f"Avg duration: {row.avg_duration_min} min"
        )
    ).add_to(m_routes)

    # Pickup marker (circle)
    folium.CircleMarker(
        location=[row.pickup_lat, row.pickup_lng],
        radius=10, color=color, fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=f"<b>Route #{i+1} PICKUP</b><br>{row.trip_count:,} trips"
    ).add_to(m_routes)

    # Dropoff marker (square)
    folium.Marker(
        location=[row.dropoff_lat, row.dropoff_lng],
        icon=folium.DivIcon(
            html=f'<div style="width:16px;height:16px;background:{color};border:2px solid white;border-radius:2px;"></div>',
            icon_size=(16, 16), icon_anchor=(8, 8)
        ),
        tooltip=f"<b>Route #{i+1} DROPOFF</b><br>{row.trip_count:,} trips"
    ).add_to(m_routes)

# Legend
legend_html = '<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:12px 16px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.25);font-size:12px;line-height:1.6;"><b>Top 5 Journeys</b><br>'
for i, row in routes_df.iterrows():
    legend_html += f'<span style="color:{colors[i]}">&#9644;</span> #{i+1}: {row.trip_count:,} trips ({row.avg_distance_mi} mi, {row.avg_duration_min} min)<br>'
legend_html += '<br>&#9679; Pickup &nbsp; &#9632; Dropoff</div>'
m_routes.get_root().html.add_child(folium.Element(legend_html))

displayHTML(m_routes._repr_html_())

In [0]:
congestion_df = spark.sql("""
SELECT
  pl.latitude, pl.longitude,
  COUNT(*) AS trip_count,
  ROUND(AVG(ft.trip_distance), 2) AS avg_distance_mi,
  ROUND(AVG(ft.trip_duration_minutes), 1) AS avg_duration_min,
  ROUND(AVG(CASE WHEN ft.trip_duration_minutes > 0 THEN ft.trip_distance / (ft.trip_duration_minutes / 60.0) END), 2) AS avg_speed_mph
FROM students_data.`team3-taxi`.fact_trip ft
INNER JOIN students_data.`team3-taxi`.dim_location pl ON ft.pickup_location_key = pl.location_key
WHERE pl.latitude BETWEEN 40.70 AND 40.88 AND pl.longitude BETWEEN -74.02 AND -73.90
  AND ft.trip_distance > 0.3
  AND ft.trip_duration_minutes > 2
GROUP BY pl.latitude, pl.longitude
HAVING COUNT(*) >= 20
ORDER BY avg_speed_mph ASC
LIMIT 5
""").toPandas()
print(f"Top {len(congestion_df)} congestion areas (slowest avg speed):")
congestion_df

In [0]:
import folium

m_congestion = folium.Map(location=[40.78, -73.97], zoom_start=13, tiles="CartoDB positron")

# Color scale: darkest red = slowest speed
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

speeds = congestion_df['avg_speed_mph'].values
min_spd, max_spd = speeds.min(), speeds.max()

for i, row in congestion_df.iterrows():
    # Intensity: slowest = biggest, darkest
    norm = 1 - (row.avg_speed_mph - min_spd) / (max_spd - min_spd + 0.01)
    radius = 15 + norm * 15
    opacity = 0.6 + norm * 0.3

    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=radius,
        color="#c0392b",
        fill=True,
        fill_color="#e74c3c",
        fill_opacity=opacity,
        tooltip=(
            f"<b>Congestion #{i+1}</b><br>"
            f"Avg speed: {row.avg_speed_mph} mph<br>"
            f"Avg duration: {row.avg_duration_min} min<br>"
            f"Avg distance: {row.avg_distance_mi} mi<br>"
            f"Trips: {row.trip_count:,}"
        )
    ).add_to(m_congestion)

    # Label
    folium.Marker(
        location=[row.latitude, row.longitude],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;font-weight:bold;color:#c0392b;text-align:center;">#{i+1}<br>{row.avg_speed_mph} mph</div>',
            icon_size=(60, 30), icon_anchor=(30, 15)
        )
    ).add_to(m_congestion)

displayHTML(m_congestion._repr_html_())

## Key Findings

### Pickup & Dropoff Hotspots
The heatmaps reveal that taxi demand in Manhattan is heavily concentrated in a handful of locations. The single busiest coordinate — near **East Harlem** (40.821°N, 73.955°W) — accounts for **1,523 pickups and 1,523 dropoffs**, suggesting a major transit hub or residential complex generating repeat trips. The next busiest point, near **Murray Hill** (40.745°N, 73.949°W), follows a similar pattern with over 1,000 trips in each direction. Pickup and dropoff distributions are strikingly symmetrical — the same locations that generate the most outbound demand also attract the most inbound traffic, which is characteristic of mixed-use commercial and transport hubs.

### Top 5 Most Common Journeys
All five of the most-repeated journeys are **short Midtown trips**, averaging just 0.8–1.4 miles and 10–14 minutes. They cluster in the corridor between **Penn Station / Herald Square** (40.751°N, 73.994°W) and **Times Square / Rockefeller Center** (40.762°N, 73.979°W), with the busiest single route recording **3,535 trips**. This pattern is consistent with commuters and business travellers making the same short hops between transit hubs, hotels, and office districts. The short distances and relatively long durations (averaging ~11 min per mile) also hint at significant Midtown congestion slowing these journeys.

Routes 1, 2, 3, and 5 all converge on the same dropoff: (40.750°N, 73.991°W). That's Penn Station / Madison Square Garden — the intersection of 7th Avenue and 31st–33rd Street.
Penn Station is the busiest rail hub in North America, serving ~600,000 daily passengers across Amtrak, NJ Transit, and the Long Island Rail Road. The pattern makes complete sense: people are being picked up from various Midtown locations (Times Square, Rockefeller Center, Herald Square) and taxied to Penn Station to catch trains. The pickups fan out across Midtown but the destination is always the same — the train station.
Route #4 is the only outlier, and notably it goes in the reverse direction — from Penn Station (40.751, -73.994) outbound to Midtown (40.759, -73.986), likely the morning arrival commuter flow heading to offices.

### Top 5 Congestion Areas
The congestion map confirms what the route durations suggest: **Midtown East is the slowest part of Manhattan for taxi travel**. The five most congested pickup zones all fall between 40.749°N–40.758°N and 73.960°W–73.994°W — roughly the rectangle bounded by **Sutton Place**, **Times Square**, **Penn Station**, and **Rockefeller Center**. Average taxi speeds here drop to just **6.9–7.3 mph**, well below the Manhattan-wide average. The worst spot, near Sutton Place / 2nd Avenue (6.94 mph), likely reflects the bottleneck of East Midtown's narrow cross-streets intersecting with FDR Drive-bound traffic.